# EXERCÍCIO 03

**Vinícius Sousa Dutra · Número USP 13686257**  

## Introdução

O objetivo é comparar o desempenho de uma rede MLP contra uma RBF na classificação da base Wine usada no projeto 01,separando os dados em treinamento (80%) e teste (20%). Iremos usar a biblioteca `scikit-learn` tanto para coletar os dados quanto para treinar as redes

In [ ]:
from typing import Final

import numpy as np
from numpy.typing import NDArray
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

type MatrizAtributos = NDArray[np.float64]
type VetorClasses = NDArray[np.int64]
type VetorReais = NDArray[np.float64]

SEED: Final[int] = 123456789
SEED=3

## Dados e partições

In [ ]:
atributos, classes = load_wine(return_X_y=True)

X = np.asarray(atributos, dtype=np.float64)
y = np.asarray(classes, dtype=np.int64)

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)


X_treino = np.asarray(X_treino, dtype=np.float64)
X_teste = np.asarray(X_teste, dtype=np.float64)
y_treino = np.asarray(y_treino, dtype=np.int64)
y_teste = np.asarray(y_teste, dtype=np.int64)

# Estas asserções são sentinelas: se a base ou a política de divisão mudar,
# preferimos uma falha explícita a continuar o relatório com dimensões inesperadas.
assert X_treino.shape == (142, 13)
assert X_teste.shape == (36, 13)

# O pequeno inventário torna visível, no próprio notebook, o resultado da divisão.
print(f"Amostras: {len(y)}")
print(f"Treinamento: {len(y_treino)}")
print(f"Teste: {len(y_teste)}")
print(f"Atributos: {X.shape[1]}")

Amostras: 178
Treinamento: 142
Teste: 36
Atributos: 13


array([[1.423e+01, 1.710e+00, 2.430e+00, ..., 1.040e+00, 3.920e+00,
        1.065e+03],
       [1.320e+01, 1.780e+00, 2.140e+00, ..., 1.050e+00, 3.400e+00,
        1.050e+03],
       [1.316e+01, 2.360e+00, 2.670e+00, ..., 1.030e+00, 3.170e+00,
        1.185e+03],
       ...,
       [1.327e+01, 4.280e+00, 2.260e+00, ..., 5.900e-01, 1.560e+00,
        8.350e+02],
       [1.317e+01, 2.590e+00, 2.370e+00, ..., 6.000e-01, 1.620e+00,
        8.400e+02],
       [1.413e+01, 4.100e+00, 2.740e+00, ..., 6.100e-01, 1.600e+00,
        5.600e+02]], shape=(178, 13))

## Padronização

Os atributos possuem escalas bastante diferentes. O `StandardScaler` aprende, apenas no treinamento, a média e o desvio padrão de cada atributo; a mesma transformação é aplicada ao teste. As duas redes recebem exatamente essas mesmas partições padronizadas.

In [3]:
# Os 13 atributos usam escalas muito diferentes. O StandardScaler aprende uma
# média e uma escala para cada coluna e transforma os atributos para uma escala
# comparável, o que é importante para modelos baseados em pesos e distâncias.
normalizador = StandardScaler()

# fit_transform aprende os parâmetros SOMENTE com o treino e já o transforma.
# O teste fica do lado de fora da sala durante o ajuste: nada de data leakage.
X_treino = np.asarray(
    normalizador.fit_transform(X_treino),
    dtype=np.float64,
)

# transform reutiliza exatamente os parâmetros aprendidos acima. Ajustar outro
# scaler no teste colocaria treino e teste em sistemas de coordenadas diferentes.
X_teste = np.asarray(normalizador.transform(X_teste), dtype=np.float64)

## Rede MLP

A MLP possui arquitetura 13–13–3: treze entradas, uma única camada intermediária com treze neurônios e três saídas. A camada intermediária usa ativação logística e os pesos são ajustados pelo solver L-BFGS do `MLPClassifier`. A largura da camada intermediária foi igualada à dimensão da entrada; não foi feita busca de hiperparâmetros.

In [ ]:
# hidden_layer_sizes recebe uma tupla com o número de neurônios de cada camada
# intermediária. A tupla de um elemento abaixo representa UMA camada com 13
# neurônios, pois X_treino.shape[1] é o número de atributos de entrada.
# A chamada a fit estima os pesos usando exclusivamente o treinamento e devolve
# o próprio classificador treinado; por isso ela pode ser encadeada aqui.
mlp = MLPClassifier(
    hidden_layer_sizes=(X_treino.shape[1],),
    # A ativação logística é aplicada aos neurônios da camada intermediária.
    activation="logistic",
    # L-BFGS é o algoritmo usado pelo scikit-learn para ajustar os pesos.
    solver="lbfgs",
    # Este é apenas um limite de segurança para o processo de otimização.
    max_iter=2_000,
    # A semente controla a inicialização aleatória dos pesos da rede.
    random_state=SEED,
).fit(X_treino, y_treino)

# score calcula a proporção de previsões corretas, isto é, a acurácia. A
# conversão para float elimina o escalar específico do NumPy da interface local.
acurácia_mlp = float(mlp.score(X_teste, y_teste))

## Rede RBF

A camada intermediária da rede RBF possui um neurônio para cada classe. Para a classe $k$, o centro $c_k$ é a média dos exemplos de treinamento daquela classe e a largura $\sigma_k$ é a raiz da média das distâncias quadráticas até o centro:

$$c_k = \frac{1}{n_k} \sum_{i:y_i=k} x_i, \qquad \sigma_k = \sqrt{\frac{1}{n_k} \sum_{i:y_i=k} \lVert x_i-c_k \rVert^2}.$$

Cada neurônio produz a ativação gaussiana

$$\phi_k(x) = \exp\left(-\frac{\lVert x-c_k \rVert^2}{2\sigma_k^2}\right).$$

As três ativações formam a entrada de uma camada de saída multiclasse, ajustada pelo `LogisticRegression`. Assim, somente a transformação radial específica do enunciado é implementada diretamente; a estimação da camada de saída permanece no scikit-learn.

In [5]:
# Esta função estima os parâmetros da camada intermediária da RBF somente a
# partir do conjunto de treinamento. Ela não altera os argumentos recebidos.
def estimar_centros_e_larguras(
    X: MatrizAtributos,
    y: VetorClasses,
) -> tuple[MatrizAtributos, VetorReais]:
    # np.unique produz as classes em ordem crescente. Para cada uma delas, a
    # máscara y == classe seleciona de X apenas os vinhos daquela classe.
    # Cada elemento de grupos tem, portanto, formato (n_k, 13).
    grupos = tuple(X[y == classe] for classe in np.unique(y))

    # mean(axis=0) calcula a média de cada atributo dentro da classe. O vetor
    # resultante é o centro c_k; stack reúne os três centros numa matriz (3, 13).
    centros = np.stack(tuple(grupo.mean(axis=0) for grupo in grupos))

    # A largura de uma classe é sua distância quadrática média até o centro,
    # seguida de uma raiz. Abrindo a expressão de dentro para fora:
    #   grupo - centro        -> deslocamento de cada amostra, por atributo;
    #   (...) ** 2            -> quadrado de cada componente do deslocamento;
    #   sum(..., axis=1)      -> distância euclidiana ao quadrado por amostra;
    #   mean(...) e sqrt(...) -> raio quadrático médio da classe.
    larguras = np.array(
        [
            np.sqrt(np.mean(np.sum((grupo - centro) ** 2, axis=1)))
            # strict=True denuncia um erro caso centros e grupos tenham tamanhos
            # diferentes, em vez de truncar silenciosamente o zip.
            for grupo, centro in zip(grupos, centros, strict=True)
        ],
        dtype=np.float64,
    )

    # Uma largura nula causaria divisão por zero na gaussiana. Com esta base isso
    # não ocorre, mas a asserção deixa explícita a pré-condição da próxima função.
    assert np.all(larguras > 0.0)

    # np.asarray conserva os valores e torna o tipo float64 inequívoco para o
    # language server. Há um centro e uma largura para cada classe.
    return np.asarray(centros, dtype=np.float64), larguras


# A transformação RBF recebe n amostras e devolve n linhas com três ativações
# gaussianas por linha: uma ativação para o centro de cada classe.
def rbf(
    X: MatrizAtributos,
    centros: MatrizAtributos,
    larguras: VetorReais,
) -> MatrizAtributos:
    # Aqui mora a parte que parece feitiçaria, mas é apenas broadcasting:
    #   X[:, None, :]       tem formato (n, 1, 13);
    #   centros[None, :, :] tem formato (1, 3, 13);
    # a subtração combina os eixos unitários e produz (n, 3, 13), contendo
    # a diferença entre cada amostra e cada centro em cada atributo.
    diferenças = X[:, np.newaxis, :] - centros[np.newaxis, :, :]

    # Somar o quadrado das diferenças no eixo dos 13 atributos deixa uma matriz
    # (n, 3): uma distância quadrática para cada par amostra-centro.
    distâncias_quadradas = np.sum(diferenças**2, axis=2)

    # larguras**2 tem formato (3,), portanto é aplicado por coluna. O resultado
    # também tem formato (n, 3), com ativações gaussianas entre zero e um.
    return np.exp(-distâncias_quadradas / (2.0 * larguras**2))


# Os centros e as larguras são estimados sem testemunhas do conjunto de teste.
centros, larguras = estimar_centros_e_larguras(X_treino, y_treino)

# A mesma transformação radial é aplicada ao treino e ao teste. A primeira matriz
# ajustará a saída; a segunda será usada apenas para medir o resultado final.
X_treino_rbf = rbf(X_treino, centros, larguras)
X_teste_rbf = rbf(X_teste, centros, larguras)

# LogisticRegression implementa a camada de saída multiclasse: ela aprende a
# combinar as três ativações radiais para escolher uma das três classes. Assim,
# reinventamos apenas a camada RBF pedida, não o otimizador da camada de saída.
saída_rbf = LogisticRegression(solver="lbfgs", max_iter=2_000).fit(
    X_treino_rbf,
    y_treino,
)

# A RBF é avaliada sobre exatamente o mesmo conjunto de teste usado pela MLP.
acurácia_rbf = float(saída_rbf.score(X_teste_rbf, y_teste))

## Resultados

As acurácias obtidas pelas duas redes sobre os 36 exemplos de teste são:

In [6]:
# O especificador .2% multiplica o valor por 100, conserva duas casas decimais
# e acrescenta o sinal de porcentagem. Nenhuma tabela cerimonial é necessária.
print(
    "Acurácia no conjunto de teste:\n"
    f"  MLP: {acurácia_mlp:.2%}\n"
    f"  RBF: {acurácia_rbf:.2%}"
)

# score deve pertencer ao intervalo fechado [0, 1]. Estas duas verificações não
# provam que os modelos estão corretos, mas capturam imediatamente saídas absurdas.
assert 0.0 <= acurácia_mlp <= 1.0
assert 0.0 <= acurácia_rbf <= 1.0

Acurácia no conjunto de teste:
  MLP: 100.00%
  RBF: 97.22%


## Conclusão

Na partição reprodutível utilizada, a MLP classificou corretamente os 36 exemplos de teste, alcançando acurácia de **100%**. A RBF classificou corretamente 35 exemplos, com acurácia de **97,22%**. Portanto, a MLP obteve o melhor resultado por uma amostra nesse conjunto de teste; ambas as redes apresentaram alta acurácia.

A comparação descreve esta divisão 80/20 específica, como solicitado, e não pretende estimar a variação dos modelos entre diferentes partições.

## Referências

- [Base Wine e `load_wine`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) — scikit-learn.
- [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) e [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) — scikit-learn.
- [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) e [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) — scikit-learn.